In [0]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, RegressionEvaluator
import numpy as np

CATALOG = "workspace"
SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

telemetry = spark.table(f"{SILVER}.telemetry_clean")
tf = spark.table(f"{SILVER}.telemetry_features")
errors = spark.table(f"{SILVER}.errors_clean")
failures = spark.table(f"{SILVER}.failures_clean")
maint = spark.table(f"{SILVER}.maint_clean")
machines = spark.table(f"{SILVER}.machines_clean")
risk = spark.table(f"{SILVER}.failure_risk_features")

In [0]:
def gold_merge(df, table_name, keys):
    full = f"{GOLD}.{table_name}"
    if not spark.catalog.tableExists(full):
        df.write.format("delta").mode("overwrite").saveAsTable(full)
    else:
        DeltaTable.forName(spark, full).alias("t").merge(
            df.alias("s"),
            " AND ".join([f"t.`{k}` <=> s.`{k}`" for k in keys])
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print("GOLD MERGED:", full)

## Q1 — Do error events predict failures within the next 24–48 hours?

In [0]:
e = errors.select("machineID","datetime","errorID")
f = failures.select("machineID", F.col("datetime").alias("failure_ts"), "failure")

q1 = (
    e.alias("e").join(
        f.alias("f"),
        (F.col("e.machineID")==F.col("f.machineID")) &
        (F.col("f.failure_ts")>F.col("e.datetime")) &
        (F.col("f.failure_ts")<=F.col("e.datetime")+F.expr("INTERVAL 48 HOURS")),
        "left"
    )
    .withColumn("hours_to_failure",
                (F.col("failure_ts").cast("long")-F.col("datetime").cast("long"))/3600)
    .withColumn("within_24h", F.when(F.col("hours_to_failure")<=24,1).otherwise(0))
    .withColumn("within_48h", F.when(F.col("hours_to_failure")<=48,1).otherwise(0))
)

q1_summary = q1.groupBy("errorID").agg(
    F.count("*").alias("error_events"),
    F.sum("within_24h").alias("failure_within_24h"),
    F.sum("within_48h").alias("failure_within_48h"),
).withColumn("rate_24h", F.col("failure_within_24h")/F.col("error_events"))  .withColumn("rate_48h", F.col("failure_within_48h")/F.col("error_events"))

gold_merge(q1_summary, "q1_error_failure_leading_indicators", ["errorID"])
display(q1_summary.orderBy(F.desc("rate_48h")))

## Q2 — How does machine age/model affect sensor baselines?

In [0]:
q2 = (
    tf.groupBy("model")
    .agg(
        F.countDistinct("machineID").alias("machines"),
        F.avg("machine_age_days").alias("avg_age_days"),
        *[F.avg(c).alias(f"avg_{c}") for c in ["volt","rotate","pressure","vibration"] if c in tf.columns],
        *[F.stddev(c).alias(f"std_{c}") for c in ["volt","rotate","pressure","vibration"] if c in tf.columns]
    )
)
gold_merge(q2, "q2_model_baseline", ["model"])

q2_age = (
    tf.withColumn(
        "age_bucket",
        F.when(F.col("machine_age_days")<30,"0-29")
         .when(F.col("machine_age_days")<90,"30-89")
         .when(F.col("machine_age_days")<180,"90-179")
         .otherwise("180+")
    )
    .groupBy("age_bucket")
    .agg(
        F.countDistinct("machineID").alias("machines"),
        *[F.avg(c).alias(f"avg_{c}") for c in ["volt","rotate","pressure","vibration"] if c in tf.columns]
    )
)
gold_merge(q2_age, "q2_age_baseline", ["age_bucket"])
display(q2)

## Q3 — Does time since maintenance affect sensor stability?

In [0]:
sensor = [c for c in ["volt","rotate","pressure","vibration"] if c in tf.columns]
q3 = tf.withColumn(
    "maintenance_bucket",
    F.when(F.col("days_since_maintenance").isNull(),"Never/Unknown")
     .when(F.col("days_since_maintenance")<=7,"0-7d")
     .when(F.col("days_since_maintenance")<=30,"8-30d")
     .otherwise("31+d")
)
q3 = q3.groupBy("maintenance_bucket").agg(
    F.count("*").alias("observations"),
    *[F.avg(F.col(f"{c}_rolling_std_24h")).alias(f"{c}_stability_std") 
      for c in sensor if f"{c}_rolling_std_24h" in tf.columns]
)
gold_merge(q3, "q3_maintenance_sensor_stability", ["maintenance_bucket"])
display(q3)

## Q4 — Forecast vibration 12 hours ahead with LSTM
This is the one deep-learning question. Run on a Databricks ML runtime with TensorFlow available. The code trains a small sequence model on a sampled machine to keep the project runnable on limited compute.

In [0]:
# Prepare one machine's chronological vibration series.
vdf = telemetry.select("machineID","datetime","vibration").dropna().orderBy("machineID","datetime")
machine_id = vdf.select("machineID").first()[0]
arr = np.array([r["vibration"] for r in vdf.filter(F.col("machineID")==machine_id).collect()], dtype=np.float32)

# 24-hour history -> 12-hour future. Dataset is hourly telemetry.
lookback, horizon = 24, 12
X, y = [], []
for i in range(len(arr)-lookback-horizon):
    X.append(arr[i:i+lookback])
    y.append(arr[i+lookback:i+lookback+horizon])
X, y = np.array(X), np.array(y)
X = X.reshape((-1, lookback, 1))

print("Training samples:", len(X), "machine:", machine_id)

In [0]:
# TensorFlow may not be installed on every Databricks runtime.
%pip install tensorflow
import tensorflow as keras_tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense

split = int(len(X)*0.8)
model = Sequential([
    LSTM(32, input_shape=(lookback,1)),
    Dense(32, activation="relu"),
    Dense(horizon)
])
model.compile(optimizer="adam", loss="mse")
history = model.fit(X[:split], y[:split], validation_split=0.1, epochs=5, batch_size=64, verbose=1)

pred = model.predict(X[split:], verbose=0)
rmse = float(np.sqrt(np.mean((pred-y[split:])**2)))
print("Q4 12-hour vibration forecast RMSE:", rmse)

In [0]:
# Store the latest 12-hour forecast as a Gold table.
last_seq = X[-1:]
forecast = model.predict(last_seq, verbose=0)[0]
q4 = spark.createDataFrame(
    [(int(machine_id), i+1, float(v), rmse) for i,v in enumerate(forecast)],
    ["machineID","hours_ahead","predicted_vibration","validation_rmse"]
)
gold_merge(q4, "q4_lstm_vibration_forecast", ["machineID","hours_ahead"])
display(q4)

## Q5 — Multisensor anomaly detection with Isolation Forest

In [0]:
# Isolation Forest through scikit-learn on a manageable sample.
from sklearn.ensemble import IsolationForest

features = [c for c in ["volt","rotate","pressure","vibration"] if c in telemetry.columns]
pdf = telemetry.select("machineID","datetime",*features).dropna().sample(False, 0.05, seed=42).toPandas()

iso = IsolationForest(n_estimators=200, contamination=0.01, random_state=42)
pdf["anomaly"] = iso.fit_predict(pdf[features])
pdf["anomaly_score"] = iso.decision_function(pdf[features])
q5 = spark.createDataFrame(pdf)

gold_merge(q5, "q5_multisensor_anomalies", ["machineID","datetime"])
display(q5.filter(F.col("anomaly")==-1).orderBy("anomaly_score").limit(50))

## Q6 — Predict failure within 24 hours

In [0]:
feature_cols = [c for c in [
    "volt","rotate","pressure","vibration","machine_age_days",
    "days_since_maintenance","volt_rolling_mean_24h","rotate_rolling_mean_24h",
    "pressure_rolling_mean_24h","vibration_rolling_mean_24h",
    "volt_rolling_std_24h","rotate_rolling_std_24h",
    "pressure_rolling_std_24h","vibration_rolling_std_24h"
] if c in risk.columns]

train = risk.select("machineID","datetime",*feature_cols,"failure_next_24h").dropna()
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
rf = RandomForestClassifier(
    labelCol="failure_next_24h", featuresCol="features",
    numTrees=100, maxDepth=8, seed=42
)
pipeline = Pipeline(stages=[assembler, rf])
train_set, test_set = train.randomSplit([0.8,0.2], seed=42)
model6 = pipeline.fit(train_set)
pred6 = model6.transform(test_set)

auc_like = pred6.select("failure_next_24h","probability","prediction")
display(pred6.select("machineID","datetime","failure_next_24h","prediction","probability").limit(20))

In [0]:
# Produce machine/time risk probabilities.
_prob1 = F.udf(lambda v: float(v[1]) if v is not None else None, "double")
q6 = pred6.select(
    "machineID","datetime",
    F.col("failure_next_24h").alias("actual_failure_next_24h"),
    "prediction",
    _prob1("probability").alias("failure_probability_24h")
)
gold_merge(q6, "q6_machine_failure_risk", ["machineID","datetime"])

## Q7 — Predict which component will fail next

In [0]:
# Build next failure component label from chronological failure history.
wf = Window.partitionBy("machineID").orderBy("datetime")
fail_next = (
    failures
    .withColumn("next_failure_comp", F.lead("failure").over(wf))
)

# Use current failure component as target when enough historical sequence exists.
# For a practical multiclass baseline, train on sensor state at failure timestamps.
q7_base = (
    failures.join(
        tf,
        (failures.machineID==tf.machineID) &
        (failures.datetime==tf.datetime),
        "left"
    )
)
target_col = "failure"
available = [c for c in ["volt","rotate","pressure","vibration","machine_age_days","days_since_maintenance"] if c in q7_base.columns]
q7_train = q7_base.select("machineID","datetime",target_col,*available).dropna()

In [0]:
q7_train = (
    failures.alias("fl").join(
        tf.alias("tf"),
        (F.col("fl.machineID")==F.col("tf.machineID")) &
        (F.col("fl.datetime")==F.col("tf.datetime")),
        "left"
    )
    .select(
        F.col("fl.machineID").alias("machineID"),
        F.col("fl.datetime").alias("datetime"),
        F.col("fl.failure").alias("failure"),
        *available
    )
    .dropna()
)
indexer = StringIndexer(inputCol="failure", outputCol="label", handleInvalid="skip")
assembler7 = VectorAssembler(inputCols=available, outputCol="features", handleInvalid="skip")
rf7 = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=100, maxDepth=8, seed=42)
pipe7 = Pipeline(stages=[indexer, assembler7, rf7])
tr7, te7 = q7_train.randomSplit([0.8,0.2], seed=42)
m7 = pipe7.fit(tr7)
p7 = m7.transform(te7)

q7 = p7.select(
    "machineID","datetime",
    F.col("failure").alias("actual_component"),
    F.col("prediction").alias("predicted_index"),
    "probability"
)
# Keep the model-index mapping as a separate table from StringIndexer labels.
labels = m7.stages[0].labels
mapping = spark.createDataFrame([(float(i), x) for i,x in enumerate(labels)], ["predicted_index","predicted_component"])
q7 = q7.join(mapping, "predicted_index", "left")
gold_merge(q7, "q7_component_predictions", ["machineID","datetime","actual_component"])
display(q7)

## Q8 — Estimate Remaining Useful Life (RUL)

In [0]:
# RUL = time from each telemetry point to the next failure of that machine.
wf_fail = Window.partitionBy("machineID").orderBy("datetime")
fail_times = failures.select("machineID","datetime").withColumnRenamed("datetime","failure_ts")

# Create a practical training target by joining each telemetry point to its next failure.
rul_join = telemetry.join(
    fail_times,
    (telemetry.machineID==fail_times.machineID) &
    (fail_times.failure_ts >= telemetry.datetime),
    "left"
)
wr = Window.partitionBy(telemetry.machineID, telemetry.datetime).orderBy(F.col("failure_ts"))
rul = (
    rul_join
    .withColumn("_rn", F.row_number().over(wr))
    .filter("_rn=1")
    .withColumn("rul_hours",
                (F.col("failure_ts").cast("long")-F.col("datetime").cast("long"))/3600)
    .filter((F.col("rul_hours")>=0) & (F.col("rul_hours")<=24*365))
    .drop("_rn","failure_ts")
)

In [0]:
rul_features = [c for c in ["volt","rotate","pressure","vibration"] if c in rul.columns]
rul_train = rul.select("machineID","datetime",*rul_features,"rul_hours").dropna()
assembler8 = VectorAssembler(inputCols=rul_features, outputCol="features", handleInvalid="skip")
rf8 = RandomForestRegressor(labelCol="rul_hours", featuresCol="features", numTrees=100, maxDepth=10, seed=42)
pipe8 = Pipeline(stages=[assembler8, rf8])
tr8, te8 = rul_train.randomSplit([0.8,0.2], seed=42)
m8 = pipe8.fit(tr8)
p8 = m8.transform(te8)

q8 = p8.select("machineID","datetime","rul_hours","prediction")
gold_merge(q8, "q8_rul_predictions", ["machineID","datetime"])
rmse8 = RegressionEvaluator(labelCol="rul_hours", predictionCol="prediction", metricName="rmse").evaluate(p8)
print("Q8 RUL RMSE:", rmse8)

## Q9 — Does maintenance reduce subsequent same-component failures?

In [0]:
# For every maintenance event, find the next failure of the same component/machine.
m9 = maint.select("machineID","datetime","comp").withColumnRenamed("datetime","maint_ts")
f9 = failures.select("machineID", F.col("datetime").alias("failure_ts"), F.col("failure").alias("failed_comp"))

q9j = m9.join(
    f9,
    (m9.machineID==f9.machineID) &
    (m9.comp==f9.failed_comp) &
    (f9.failure_ts>m9.maint_ts),
    "left"
)
w9 = Window.partitionBy("machineID","maint_ts","comp").orderBy("failure_ts")
q9 = (
    q9j.withColumn("_rn", F.row_number().over(w9))
    .filter("_rn=1")
    .withColumn("days_to_next_same_component_failure",
                F.datediff(F.to_date("failure_ts"), F.to_date("maint_ts")))
    .select("machineID","maint_ts","comp","failure_ts","days_to_next_same_component_failure")
)
gold_merge(q9, "q9_maintenance_effectiveness", ["machineID","maint_ts","comp"])
display(q9.orderBy("days_to_next_same_component_failure"))

## Q10 — MTBF by machine model and component

In [0]:
# MTBF is calculated from consecutive failures of the same machine/component.
f10 = (
    failures.join(machines.select("machineID","model"), "machineID", "left")
    .withColumn("component", F.col("failure"))
)
w10 = Window.partitionBy("machineID","component").orderBy("datetime")
q10 = (
    f10
    .withColumn("previous_failure", F.lag("datetime").over(w10))
    .withColumn("hours_between_failures",
                (F.col("datetime").cast("long")-F.col("previous_failure").cast("long"))/3600)
    .groupBy("model","component")
    .agg(
        F.count("*").alias("failure_events"),
        F.avg("hours_between_failures").alias("mtbf_hours"),
        F.avg("hours_between_failures")/24.0
            .alias("mtbf_days")
    )
)
gold_merge(q10, "q10_mtbf_model_component", ["model","component"])
display(q10.orderBy("model","component"))

## Final project summary

The Gold layer now contains answer-ready tables for all ten questions.

### Important limitation for Q10
The supplied maintenance dataset does **not** contain a maintenance-team, technician, or repair-type field. Therefore team/repair-type MTBF cannot be calculated from the supplied data without inventing information. The notebook reports model/component MTBF instead.

### Main Databricks concepts demonstrated
- Unity Catalog three-part naming
- Databricks Volume
- Delta Lake
- Bronze/Silver/Gold Medallion Architecture
- `MERGE INTO` / Delta upserts
- Incremental/idempotent processing
- Window functions
- Feature engineering
- LSTM time-series forecasting
- Isolation Forest anomaly detection
- Random Forest classification
- Random Forest regression
- Gold analytical tables

In [0]:
# Final Gold inventory
display(spark.sql(f"SHOW TABLES IN {GOLD}"))